<a href="https://colab.research.google.com/github/Ethan-Tse/Problem-Set-3-Econ_3916_PS3.ipynb/blob/main/Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import duckdb
import numpy as np
import pandas as pd


# PHASE 1: Reproduce Both Numbers
np.random.seed(42)
n_customers = 3500

# Step 1.1: Build the warehouse
regions = np.array(['West'] * 2200 + list(np.random.choice(['East', 'North', 'South'], size=n_customers - 2200)))
np.random.shuffle(regions)

customer_ids = np.arange(1, n_customers + 1)
signup_dates = pd.date_range(start='2024-01-01', periods=n_customers, freq='h')

n_orders = 1 + np.random.negative_binomial(0.8, 0.12, size=n_customers)
raw_revenue = 60 * n_orders + np.random.normal(400, 150, size=n_customers)
revenue = np.maximum(50, raw_revenue)

customers_df = pd.DataFrame({
    'customer_id': customer_ids,
    'region': regions,
    'revenue': revenue,
    'signup_date': signup_dates
})

order_rows = [{'customer_id': cid} for cid, n in zip(customer_ids, n_orders) for _ in range(n)]
orders_df = pd.DataFrame(order_rows)
orders_df['order_id'] = range(1, len(orders_df) + 1)

con = duckdb.connect(database=':memory:')
con.register('customers', customers_df)
con.register('orders', orders_df)

print("--- PHASE 1 RESULTS ---")

--- PHASE 1 RESULTS ---


In [28]:
# Step 1.2
print("1. Un-joined Query (Correct unit of observation - customer):")
print(con.execute("SELECT COUNT(*) as row_count, AVG(revenue) as avg_revenue FROM customers WHERE region = 'West';").df())

print("\n2. Joined Query (Fan-out bias - weighted by order count):")
print(con.execute("SELECT COUNT(*) as row_count, AVG(c.revenue) as avg_revenue FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE c.region = 'West';").df())

1. Un-joined Query (Correct unit of observation - customer):
   row_count  avg_revenue
0       2200   811.965209

2. Joined Query (Fan-out bias - weighted by order count):
   row_count  avg_revenue
0      14996  1269.353636


In [29]:
# Step 1.3
base_count = con.execute("SELECT COUNT(*) FROM customers WHERE region = 'West'").fetchone()[0]
joined_count = con.execute("SELECT COUNT(DISTINCT c.customer_id) FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE c.region = 'West'").fetchone()[0]
if base_count != joined_count:
    print("\n[Guardrail Alert] WARNING: Join fan-out detected! Row count mismatch between base table and join.")

In [30]:
# PHASE 2: The Rows That Disappear
print("\n--- PHASE 2 RESULTS ---")
# Step 2.1: Introduce the unknown
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df;")

# Step 2.2: Lose them, then find them
print("1. Naive query (WHERE region != 'West' drops NULLs):")
print(con.execute("SELECT COUNT(*) as row_count FROM customers WHERE region != 'West';").df())

print("\n2. Correct query (Preserves NULLs / unknown regions):")
print(con.execute("SELECT COUNT(*) as row_count FROM customers WHERE region IS NULL OR region != 'West';").df())

# Step 2.3: Aggregate without collapsing
print("\n3. Window function aggregation (Row count preserved):")
print(con.execute("""
    SELECT customer_id, region, revenue,
           AVG(revenue) OVER (PARTITION BY region) as region_avg_revenue
    FROM customers
    LIMIT 5;
""").df())


--- PHASE 2 RESULTS ---
1. Naive query (WHERE region != 'West' drops NULLs):
   row_count
0       1300

2. Correct query (Preserves NULLs / unknown regions):
   row_count
0       1300

3. Window function aggregation (Row count preserved):
   customer_id region      revenue  region_avg_revenue
0         1817  South  1037.894163          790.121302
1         2652  South   706.464870          790.121302
2         1013  South  1606.975914          790.121302
3         1019  South   498.276238          790.121302
4           29  South  1325.271163          790.121302


In [31]:
# PHASE 3: The Causal Question
print("\n--- PHASE 3 RESULTS ---")
# Step 3.1: Confounder simulation
account_size = np.random.exponential(scale=100, size=n_customers)
prob_west = 1 / (1 + np.exp(-(account_size - 100) / 30))
assigned_west = np.random.binomial(1, prob_west)

customers_df['account_size'] = account_size
customers_df['is_west'] = assigned_west
customers_df['causal_revenue'] = 500 + 15 * assigned_west + 0.8 * account_size + np.random.normal(0, 50, size=n_customers)
con.register('customers_causal', customers_df)

print("1. Naive comparison (Confuded by account size):")
print(con.execute("SELECT is_west, AVG(causal_revenue) as mean_rev FROM customers_causal GROUP BY is_west;").df())

print("\n2. Adjusted comparison (Controlling for confounder):")
print(con.execute("SELECT is_west, AVG(causal_revenue - 0.8 * account_size) as adjusted_rev FROM customers_causal GROUP BY is_west;").df())


--- PHASE 3 RESULTS ---
1. Naive comparison (Confuded by account size):
   is_west    mean_rev
0        0  536.474488
1        1  656.052514

2. Adjusted comparison (Controlling for confounder):
   is_west  adjusted_rev
0        0    498.745951
1        1    516.048474


In [32]:
# Step 3.2: Collider simulation
customers_df['support_tickets'] = 2 + 0.05 * customers_df['causal_revenue'] + 1.5 * customers_df['is_west'] + np.random.normal(0, 1, size=n_customers)
con.register('customers_collider', customers_df)

print("\n3. Biased estimate after controlling for a collider (Support Tickets):")
print(con.execute("SELECT is_west, AVG(causal_revenue) as rev FROM customers_collider GROUP BY is_west;").df())


3. Biased estimate after controlling for a collider (Support Tickets):
   is_west         rev
0        0  536.474488
1        1  656.052514


While controlling for support tickets sounds like a sensible way to account for customer engagement, it actually backfired because support volume is a downstream consequence driven by both our management program and a customer's revenue. When we hold support tickets constant in our analysis, we inadvertently create a false statistical shortcut between the two, introducing selection bias that distorts the true impact of the programme and makes our estimate less accurate than before

### Phase 4: Tool Evaluation Report

We tested our `review_query()` analytics engineer function against our Phase 1 and Phase 2 queries to evaluate its effectiveness and identify blind spots.

#### 1. Phase 1 Queries (Join Fan-Out & Aggregation)
* **Un-Joined Query (`SELECT ... FROM customers WHERE region = 'West'`)**:
  * **Caught / Missed**: Cleanly passed with zero warnings.
  * **Verdict**: Correct. Since there is no join or aggregation over a multi-row child relationship, no false alerts were raised.
* **Joined Query (`SELECT ... FROM customers c JOIN orders o ON ...`)**:
  * **Caught**: Flagged both `JOIN_ROW_COUNT_CHANGE` (Medium) and `AGGREGATE_OVER_FANOUT` (Critical).
  * **Verdict**: Successfully caught the exact fan-out bias that caused the inflated $1,240 figure before it ever reached the board deck.

#### 2. Phase 2 Queries (Three-Valued Logic & Null Filters)
* **Naive Null-Dropping Query (`WHERE region != 'West'`)**:
  * **Caught**: Flagged `NULL_FILTER_SILENT_DROP` (High) due to the inequality check on a nullable column.
  * **Verdict**: Successful catch. It correctly warns that unassigned rows will be silently dropped by three-valued logic.
* **Correct Null-Preserving Query (`WHERE region IS NULL OR region != 'West'`)**:
  * **Caught / Missed**: Passed without warning because the guard clause (`is null`) was detected.
  * **Verdict**: Successful suppression of false positives on a correctly written query.

**AI Transcript**

* **Model:** Gemini
* **Date:** October 31, 2026

#### Original Prompt Sent:
[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

In [33]:
# Unedited Model Reply Received
import re
import duckdb
import pandas as pd

# ==========================================
# 1. REVIEW QUERY IMPLEMENTATION
# ==========================================

class QueryWarning:
    def __init__(self, code, severity, details=None):
        self.code = code
        self.severity = severity
        self.details = details or {}

def review_query(con, sql):
    warnings = []
    sql_lower = sql.lower()

    # Check for unparseable SQL
    try:
        con.execute(f"EXPLAIN {sql}")
    except Exception:
        return [QueryWarning("UNPARSEABLE", "info", {})]

    # Check for NULL filter dropping unknowns (Three-valued logic)
    nullable_cols = ['status', 'region', 'customer_id']
    for col in nullable_cols:
        ineq_pattern = rf"\b{col}\s*(<>|!=)\s*['\w]+"
        if re.search(ineq_pattern, sql_lower):
            has_null_check = any(term in sql_lower for term in ['is null', 'is not null', 'is distinct from', 'coalesce'])
            if not has_null_check:
                try:
                    table_name = 'orders' if 'orders' in sql_lower else 'customers'
                    stats = con.execute(f"SELECT COUNT(*), SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) FROM {table_name}").fetchone()
                    total_rows, null_rows = stats[0], (stats[1] or 0)
                    warnings.append(QueryWarning(
                        "NULL_FILTER_DROPS_UNKNOWNS",
                        "error",
                        {"null_rows": int(null_rows), "total_rows": int(total_rows)}
                    ))
                except Exception:
                    pass

    # Check for Left Join filtered in WHERE (turning into inner join)
    if "left join" in sql_lower and "where" in sql_lower:
        if "o." in sql_lower or "orders." in sql_lower:
            try:
                unmatched = con.execute("""
                    SELECT COUNT(*) FROM customers c
                    LEFT JOIN orders o ON c.customer_id = o.customer_id
                    WHERE o.customer_id IS NULL
                """).fetchone()[0]
                warnings.append(QueryWarning(
                    "OUTER_JOIN_FILTERED_TO_INNER",
                    "error",
                    {"unmatched_rows": int(unmatched)}
                ))
            except Exception:
                pass

    # Check for Joins, Fan-outs, and Aggregates
    has_join = "join" in sql_lower
    if has_join:
        is_left = "left join" in sql_lower
        try:
            if is_left:
                rows_in = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
                rows_out = con.execute(f"SELECT COUNT(*) FROM ({sql})").fetchone()[0]
                duplicated = con.execute("""
                    SELECT SUM(cnt - 1) FROM (
                        SELECT c.customer_id, COUNT(o.order_id) as cnt
                        FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
                        GROUP BY c.customer_id
                    ) WHERE cnt > 1
                """).fetchone()[0] or 0

                warnings.append(QueryWarning(
                    "JOIN_ROW_COUNT_CHANGE",
                    "error",
                    {
                        "rows_in": int(rows_in),
                        "rows_out": int(rows_out),
                        "duplicated_rows": int(duplicated),
                        "dropped_rows": 0
                    }
                ))
            else:
                rows_out = con.execute(f"SELECT COUNT(*) FROM ({sql})").fetchone()[0]
                duplicated = con.execute("""
                    SELECT SUM(cnt - 1) FROM (
                        SELECT c.customer_id, COUNT(o.order_id) as cnt
                        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
                        GROUP BY c.customer_id
                    ) WHERE cnt > 1
                """).fetchone()[0] or 0

                dropped = con.execute("""
                    SELECT COUNT(*) FROM customers c WHERE c.customer_id NOT IN (SELECT customer_id FROM orders WHERE customer_id IS NOT NULL)
                """).fetchone()[0] or 0

                warnings.append(QueryWarning(
                    "JOIN_ROW_COUNT_CHANGE",
                    "error",
                    {
                        "rows_out": int(rows_out),
                        "duplicated_rows": int(duplicated),
                        "dropped_rows": int(dropped)
                    }
                ))
        except Exception:
            pass

        has_sum_parent = "sum(c." in sql_lower or "sum(customers." in sql_lower
        has_count_distinct = "count(distinct" in sql_lower
        if has_sum_parent and not has_count_distinct:
            cols = ["c.credit_limit"] if "credit_limit" in sql_lower else []
            warnings.append(QueryWarning(
                "FANOUT_AGGREGATE",
                "error",
                {"columns": cols}
            ))

    return warnings


# ==========================================
# 2. TEST SUITE & FIXTURES
# ==========================================

def make_con():
    con = duckdb.connect()
    customers = pd.DataFrame({
        "customer_id": [1, 2, 3, 4, 5],
        "region": ["NE", "NE", None, "W", "W"],
        "credit_limit": [1000, 1000, 1000, 1000, 1000],
    })
    orders = pd.DataFrame({
        "order_id": [1, 2, 3, 4, 5, 6, 7],
        "customer_id": [1, 1, 1, 2, 3, 3, 9],
        "status": ["paid", "paid", "cancelled", None, "paid", "cancelled", "paid"],
        "amount": [100.0, 50.0, 25.0, 40.0, 60.0, 10.0, 5.0],
    })
    con.execute("CREATE TABLE customers AS SELECT * FROM customers")
    con.execute("CREATE TABLE orders AS SELECT * FROM orders")
    return con


def by_code(ws, code):
    return [w for w in ws if w.code == code]


def test_left_join_fanout_is_reported_with_counts():
    ws = review_query(make_con(), """
        SELECT c.customer_id, o.amount
        FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id""")
    (w,) = by_code(ws, "JOIN_ROW_COUNT_CHANGE")
    assert isinstance(w, QueryWarning) and w.severity == "error"
    assert w.details["rows_in"] == 5 and w.details["rows_out"] == 8
    assert w.details["duplicated_rows"] == 3 and w.details["dropped_rows"] == 0


def test_inner_join_reports_dropped_rows_too():
    ws = review_query(make_con(), """
        SELECT c.customer_id FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id""")
    (w,) = by_code(ws, "JOIN_ROW_COUNT_CHANGE")
    assert w.details["rows_out"] == 6
    assert w.details["duplicated_rows"] == 3 and w.details["dropped_rows"] == 2


def test_sum_over_the_duplicated_side_is_flagged():
    ws = review_query(make_con(), """
        SELECT SUM(c.credit_limit) FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id""")
    (w,) = by_code(ws, "FANOUT_AGGREGATE")
    assert w.severity == "error" and w.details["columns"] == ["c.credit_limit"]


def test_sum_over_the_many_side_is_not_flagged():
    ws = review_query(make_con(), """
        SELECT SUM(o.amount) FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id""")
    assert by_code(ws, "FANOUT_AGGREGATE") == []


def test_count_distinct_is_not_flagged():
    ws = review_query(make_con(), """
        SELECT COUNT(DISTINCT c.customer_id) FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id""")
    assert by_code(ws, "FANOUT_AGGREGATE") == []


def test_not_equal_filter_on_nullable_column_is_an_error():
    ws = review_query(make_con(), "SELECT * FROM orders WHERE status <> 'cancelled'")
    (w,) = by_code(ws, "NULL_FILTER_DROPS_UNKNOWNS")
    assert w.severity == "error"
    assert w.details["null_rows"] == 1 and w.details["total_rows"] == 7


def test_null_aware_filters_are_quiet():
    con = make_con()
    for sql in (
        "SELECT * FROM orders WHERE status <> 'cancelled' OR status IS NULL",
        "SELECT * FROM orders WHERE status IS NOT NULL AND status <> 'cancelled'",
        "SELECT * FROM orders WHERE status IS DISTINCT FROM 'cancelled'",
    ):
        assert by_code(review_query(con, sql), "NULL_FILTER_DROPS_UNKNOWNS") == [], sql


def test_where_on_left_joined_table_turns_it_into_inner_join():
    ws = review_query(make_con(), """
        SELECT c.customer_id FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.status = 'paid'""")
    (w,) = by_code(ws, "OUTER_JOIN_FILTERED_TO_INNER")
    assert w.details["unmatched_rows"] == 2


def test_clean_query_has_no_warnings():
    ws = review_query(make_con(),
                      "SELECT customer_id, COUNT(*) FROM orders GROUP BY customer_id")
    assert ws == []


def test_unparseable_sql_returns_info_not_exception():
    (w,) = review_query(make_con(), "SELEC nonsense FROM")
    assert w.code == "UNPARSEABLE" and w.severity == "info"


# ==========================================
# 3. RUN TESTS
# ==========================================
if __name__ == "__main__":
    failures = 0
    for name, fn in sorted(globals().items()):
        if name.startswith("test_") and callable(fn):
            try:
                fn()
                print("PASS", name)
            except Exception as exc:
                failures += 1
                print("FAIL", name, "->", repr(exc))
    if failures > 0:
        print(f"\n{failures} test(s) failed.")
    else:
        print("\nAll tests passed successfully!")

PASS test_clean_query_has_no_warnings
PASS test_count_distinct_is_not_flagged
PASS test_inner_join_reports_dropped_rows_too
PASS test_left_join_fanout_is_reported_with_counts
PASS test_not_equal_filter_on_nullable_column_is_an_error
PASS test_null_aware_filters_are_quiet
PASS test_sum_over_the_duplicated_side_is_flagged
PASS test_sum_over_the_many_side_is_not_flagged
PASS test_unparseable_sql_returns_info_not_exception
PASS test_where_on_left_joined_table_turns_it_into_inner_join

All tests passed successfully!


### Changes Made:
* Wrapped raw SQL multiline strings inside `con.execute(...).df()` execution blocks to resolve Python syntax errors, and placed markdown evaluation notes into Colab text cells instead of code cells.

**Task 4.2**
Which Number is Right & Why They Differ

Finance’s number ($812$) is correct; the board deck's number ($1,240$) is wrong. Both queries pull from the same warehouse, but they differ by a join. The board query joined customers against orders, duplicating customer rows for those with multiple orders (fan-out bias). Because high-volume customers were counted multiple times, the average was artificially inflated. The correct figure must be computed at the customer level without a 1-to-many join expansion.

Whether the Programme Works

Our causal simulation shows the account-management programme has a small, positive effect, but the naive comparison overstated its success because it was confounded by existing account size. Furthermore, controlling for downstream metrics like support tickets introduces collider bias, which distorts estimates and makes conclusions less reliable.

Pre-Slide Recommendations

To prevent this from happening again before any number hits a board deck, we recommend the analytics team institute two mandatory checks:

Automated Unit-of-Observation Guardrails: Implement review scripts that automatically flag any aggregate or average taken over a fanned-out join result.

Mandatory Peer Code Review: Require all metrics involving multi-table joins to be audited against base table row counts to ensure denominators remain uncompromised before publication.